In [2]:
!pip install pypdf sentence-transformers faiss-cpu numpy transformers torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 45.3 MB/s eta 0:00:00


In [4]:
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from transformers import pipeline
from typing import List
from textwrap import wrap as word_wrap

In [5]:
# Load PDF
def load_pdf(path: str) -> str:
    reader = PdfReader(path)
    text = ""
    for page in reader.pages:
        text += page.extract_text() or ""
    return text

In [6]:
# Chunk Text
def chunk_text(text: str, chunk_size=500, overlap=50) -> List[str]:
    words = text.split()
    chunks = []
    start = 0

    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap

    return chunks

In [7]:
# Build Embeddings
def build_embeddings(chunks):
    print("\n[+] Loading embedding model...")
    model = SentenceTransformer("all-MiniLM-L6-v2")

    print(f"[+] Encoding {len(chunks)} chunks...")
    embeddings = model.encode(chunks).astype("float32")

    return model, embeddings

In [8]:
# Build FAISS Index
def build_faiss_index(embeddings):
    dim = embeddings.shape[1]

    index = faiss.IndexFlatL2(dim)
    index.add(embeddings)

    print(f"[+] FAISS index ready with {index.ntotal} vectors")
    return index

In [9]:
# Retrieve
def retrieve(query, model, index, chunks, top_k=5):
    query_vec = model.encode([query]).astype("float32")

    distances, indices = index.search(query_vec, top_k)

    results = [chunks[i] for i in indices[0] if i != -1]
    return results

In [10]:
# Generate Answer
def generate_answer(query, context_chunks, generator):
    context = "\n\n---\n\n".join(context_chunks)

    prompt = f"""
Answer the question using ONLY the context below.
If the answer is not in the context, say:
"I don't know based on the provided documents."

Context:
{context}

Question: {query}
Answer:
"""

    result = generator(prompt, max_new_tokens=300)
    return result[0]["generated_text"]

In [11]:
# Build RAG Pipeline
def build_rag_pipeline(pdf_files):
    all_chunks = []

    for file in pdf_files:
        print(f"\n[+] Loading {file}")
        text = load_pdf(file)

        chunks = chunk_text(text)
        print(f" -> {len(chunks)} chunks")

        all_chunks.extend(chunks)

    print(f"\n[+] Total chunks: {len(all_chunks)}")

    model, embeddings = build_embeddings(all_chunks)
    index = build_faiss_index(embeddings)

    return model, index, all_chunks

In [12]:
# Ask Function
def ask(query, model, index, chunks, generator):
    print("\n" + "="*60)
    print("QUESTION:", query)
    print("="*60)

    context = retrieve(query, model, index, chunks)

    print(f"\n[+] Retrieved {len(context)} chunks")

    answer = generate_answer(query, context, generator)

    print("\ ANSWER:\n")
    for line in word_wrap(answer, 80):
        print(line)

    return answer

In [14]:
# Upload PDFs
from google.colab import files

print("Upload your PDF files:")
uploaded = files.upload()

PDF_FILES = list(uploaded.keys())


# Run Pipeline
embedder, index, chunks = build_rag_pipeline(PDF_FILES)

print("\n[+] Loading FLAN-T5 model...")
generator = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

Upload your PDF files:


Saving 5b_pdf.pdf to 5b_pdf (1).pdf
Saving 5b_pdf2.pdf to 5b_pdf2 (1).pdf

[+] Loading 5b_pdf (1).pdf
 -> 14 chunks

[+] Loading 5b_pdf2 (1).pdf
 -> 23 chunks

[+] Total chunks: 37

[+] Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[+] Encoding 37 chunks...
[+] FAISS index ready with 37 vectors

[+] Loading FLAN-T5 model...


...

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausalLM', 'FalconForCausalLM', 'FalconH1ForCausalLM', 'FalconMambaForCausa

In [15]:
# Interactive Loop
print("\nSystem Ready! Type 'exit' to stop.\n")

while True:
    query = input("👉 Ask a question: ")

    if query.lower() == "exit":
        print("👋 Exiting...")
        break

    ask(query, embedder, index, chunks, generator)


System Ready! Type 'exit' to stop.

👉 Ask a question: What is the main idea of the attention mechanism?


Token indices sequence length is longer than the specified maximum sequence length for this model (3612 > 512). Running this sequence through the model will result in indexing errors
Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.



QUESTION: What is the main idea of the attention mechanism?

[+] Retrieved 5 chunks


Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



💡 ANSWER:

 Answer the question using ONLY the context below. If the answer is not in the
context, say: "I don't know based on the provided documents."  Context: and W O
∈ Rhdv×dmodel. In this work we employ h = 8 parallel attention layers, or heads.
For each of these we use dk = dv = dmodel/h = 64. Due to the reduced dimension
of each head, the total computational cost is similar to that of single-head
attention with full dimensionality. 3.2.3 Applications of Attention in our Model
The Transformer uses multi-head attention in three different ways: • In
"encoder-decoder attention" layers, the queries come from the previous decoder
layer, and the memory keys and values come from the output of the encoder. This
allows every position in the decoder to attend over all positions in the input
sequence. This mimics the typical encoder-decoder attention mechanisms in
sequence-to-sequence models such as [38, 2, 9]. • The encoder contains self-
attention layers. In a self-attention layer all of

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



💡 ANSWER:

 Answer the question using ONLY the context below. If the answer is not in the
context, say: "I don't know based on the provided documents."  Context: for
measuring readability. Journalism Bulletin, 30(4):415–433. Erik F Tjong Kim Sang
and Fien De Meulder. 2003. Introduction to the conll-2003 shared task: Language-
independent named entity recognition. In CoNLL. Joseph Turian, Lev Ratinov, and
Yoshua Bengio. 2010. Word representations: A simple and general method for semi-
supervised learning. In Proceedings of the 48th Annual Meeting of the
Association for Compu- tational Linguistics, ACL ’10, pages 384–394. Ashish
Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N Gomez,
Lukasz Kaiser, and Illia Polosukhin. 2017. Attention is all you need. In
Advances in Neural Information Pro- cessing Systems, pages 6000–6010. Pascal
Vincent, Hugo Larochelle, Yoshua Bengio, and Pierre-Antoine Manzagol. 2008.
Extracting and composing robust features with denoising a